# UA-DETRAC MOTA Tracking Metrics Evaluation (Modular Edition)

Notebook này được thiết kế như một **module độc lập** chuyên dùng để đánh giá hiệu năng của các thuật toán Multi-Object Tracking (MOT) trên bộ dataset UA-DETRAC.

## 📂 Cách thức hoạt động:
Notebook này sẽ:
1. **Đọc kết quả dự đoán (Tracks Predictions)**: Từ các file `.txt` đầu ra của bất kỳ thuật toán nào (SORT, DeepSORT, ByteTrack...) được lưu trong một thư mục chỉ định dưới định dạng chuẩn MOT-Challenge (`<frame>,<id>,<left>,<top>,<width>,<height>,...`).
2. **Tải Ground-Truth XML**: Từ thư mục Annotations của UA-DETRAC.
3. **Tính toán các chỉ số chuẩn MOT**:
   * **MOTA** (Multiple Object Tracking Accuracy - Độ chính xác tracking).
   * **FP** (False Positives - Nhận diện sai/thừa).
   * **FN** (False Negatives - Bỏ sót xe).
   * **ID Switches** (Số lần tráo đổi/lệch ID của xe).
   * **GT Count** (Tổng số xe thực tế trong ground-truth).

## 🎯 Ưu điểm của Thiết kế Modular:
- Bạn có thể dùng chung Notebook này cho **nhiều thuật toán khác nhau** mà không cần sửa code của thuật toán đó.
- Tách biệt hoàn toàn phần tracking và phần tính toán metrics để tối ưu hóa bộ nhớ và tốc độ xử lý.

In [ ]:
import os
import re
import sys
from glob import glob
from pathlib import Path
from typing import List, Dict
import xml.etree.ElementTree as ET

import cv2
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
from tqdm.auto import tqdm

print("✓ Môi trường đánh giá Metrics đã tải thành công!")


## 1. Định nghĩa các Hàm Trợ giúp & Hungarian Matching

In [ ]:
def find_annotation_xml(sequence_name: str, annotations_folder: str = None) -> str:
    """Tìm file XML ground-truth chứa thông tin Ignored Region và Ground-Truth Bbox."""
    if not sequence_name or not annotations_folder:
        return None
    candidates = [
        os.path.join(annotations_folder, f"{sequence_name}.xml"),
        os.path.join(annotations_folder, sequence_name, f"{sequence_name}.xml"),
    ]
    for path in candidates:
        if os.path.isfile(path):
            return os.path.abspath(path)
    return None


def load_detrac_gt_xml(xml_path: str) -> dict:
    """Đọc ground-truth từ file XML thành định dạng {frame_num: [(target_id, bbox), ...]}."""
    gt_by_frame = {}
    if not xml_path or not os.path.isfile(xml_path):
        return gt_by_frame
    root = ET.parse(xml_path).getroot()
    for frame_node in root.findall("frame"):
        frame_num = int(frame_node.attrib.get("num", -1))
        objects = []
        target_list = frame_node.find("target_list")
        if target_list is not None:
            for target_node in target_list.findall("target"):
                box_node = target_node.find("box")
                if box_node is None:
                    continue
                target_id = int(target_node.attrib["id"])
                left = float(box_node.attrib["left"])
                top = float(box_node.attrib["top"])
                width = float(box_node.attrib["width"])
                height = float(box_node.attrib["height"])
                bbox = np.array([left, top, left + width, top + height], dtype=float)
                objects.append((target_id, bbox))
        gt_by_frame[frame_num] = objects

    return gt_by_frame


def load_detrac_ignored_regions_xml(xml_path: str) -> List[np.ndarray]:
    """Đọc các Ignored Region từ file XML."""
    ignored_regions = []
    if not xml_path or not os.path.isfile(xml_path):
        return ignored_regions
    root = ET.parse(xml_path).getroot()
    ignored_node = root.find("ignored_region")
    if ignored_node is None:
        return ignored_regions
    for box_node in ignored_node.findall("box"):
        left = float(box_node.attrib["left"])
        top = float(box_node.attrib["top"])
        width = float(box_node.attrib["width"])
        height = float(box_node.attrib["height"])
        ignored_regions.append(
            np.array([left, top, left + width, top + height], dtype=float)
        )
    return ignored_regions


def bbox_area(bbox: np.ndarray) -> float:
    return max(0.0, float(bbox[2] - bbox[0])) * max(0.0, float(bbox[3] - bbox[1]))


def intersection_area(box_a: np.ndarray, box_b: np.ndarray) -> float:
    x1 = max(float(box_a[0]), float(box_b[0]))
    y1 = max(float(box_a[1]), float(box_b[1]))
    x2 = min(float(box_a[2]), float(box_b[2]))
    y2 = min(float(box_a[3]), float(box_b[3]))
    return max(0.0, x2 - x1) * max(0.0, y2 - y1)


def ignored_overlap_ratio(bbox: np.ndarray, ignored_regions: List[np.ndarray]) -> float:
    area = bbox_area(bbox)
    if area <= 0.0 or not ignored_regions:
        return 0.0
    ignored_area = sum(intersection_area(bbox, region) for region in ignored_regions)
    return min(1.0, ignored_area / area)


def filter_tracks_by_ignore(
    tracks: List[tuple[int, np.ndarray]],
    ignored_regions: List[np.ndarray],
    overlap_threshold: float,
):
    if not tracks or not ignored_regions:
        return tracks
    return [
        (track_id, bbox)
        for track_id, bbox in tracks
        if ignored_overlap_ratio(bbox, ignored_regions) < overlap_threshold
    ]


def iou(bb_test: np.ndarray, bb_gt: np.ndarray) -> float:
    xx1 = np.maximum(bb_test[0], bb_gt[0])
    yy1 = np.maximum(bb_test[1], bb_gt[1])
    xx2 = np.minimum(bb_test[2], bb_gt[2])
    yy2 = np.minimum(bb_test[3], bb_gt[3])
    w = np.maximum(0.0, xx2 - xx1)
    h = np.maximum(0.0, yy2 - yy1)
    inter = w * h
    area1 = (bb_test[2] - bb_test[0]) * (bb_test[3] - bb_test[1])
    area2 = (bb_gt[2] - bb_gt[0]) * (bb_gt[3] - bb_gt[1])
    union = area1 + area2 - inter
    return inter / union if union > 0.0 else 0.0


## 2. Tính toán MOTA Metric Core

In [ ]:
def _match_gt_to_preds(
    gt_bboxes: List[np.ndarray],
    pred_bboxes: List[np.ndarray],
    iou_threshold: float = 0.5,
):
    if len(gt_bboxes) == 0 and len(pred_bboxes) == 0:
        return [], [], []
    if len(gt_bboxes) == 0:
        return [], [], list(range(len(pred_bboxes)))
    if len(pred_bboxes) == 0:
        return [], list(range(len(gt_bboxes))), []

    iou_matrix = np.zeros((len(gt_bboxes), len(pred_bboxes)), dtype=float)
    for g_idx, gt_bbox in enumerate(gt_bboxes):
        for p_idx, pred_bbox in enumerate(pred_bboxes):
            iou_matrix[g_idx, p_idx] = iou(gt_bbox, pred_bbox)

    row_ind, col_ind = linear_sum_assignment(-iou_matrix)
    matches = []
    unmatched_gts = list(range(len(gt_bboxes)))
    unmatched_preds = list(range(len(pred_bboxes)))

    for g_idx, p_idx in zip(row_ind, col_ind):
        if iou_matrix[g_idx, p_idx] < iou_threshold:
            continue
        matches.append((g_idx, p_idx))
        unmatched_gts.remove(g_idx)
        unmatched_preds.remove(p_idx)

    return matches, unmatched_gts, unmatched_preds


def compute_mota(
    gt_frames: List[List[tuple[int, np.ndarray]]],
    pred_frames: List[List[tuple[int, np.ndarray]]],
    iou_threshold: float = 0.5,
) -> dict:
    total_gt = 0
    false_negatives = 0
    false_positives = 0
    id_switches = 0
    gt_to_last_pred = {}

    for gt_objs, pred_objs in zip(gt_frames, pred_frames):
        gt_ids = [obj[0] for obj in gt_objs]
        gt_bboxes = [obj[1] for obj in gt_objs]
        pred_ids = [obj[0] for obj in pred_objs]
        pred_bboxes = [obj[1] for obj in pred_objs]
        total_gt += len(gt_ids)
        matches, unmatched_gts, unmatched_preds = _match_gt_to_preds(
            gt_bboxes, pred_bboxes, iou_threshold
        )
        false_negatives += len(unmatched_gts)
        false_positives += len(unmatched_preds)

        for gt_idx, pred_idx in matches:
            gt_id = gt_ids[gt_idx]
            pred_id = pred_ids[pred_idx]
            last_pred_id = gt_to_last_pred.get(gt_id)
            if last_pred_id is None:
                gt_to_last_pred[gt_id] = pred_id
            elif pred_id != last_pred_id:
                id_switches += 1
                gt_to_last_pred[gt_id] = pred_id

    mota = (
        1.0 - (false_negatives + false_positives + id_switches) / total_gt
        if total_gt > 0
        else float("nan")
    )
    return {
        "MOTA": mota,
        "FP": false_positives,
        "FN": false_negatives,
        "ID_switches": id_switches,
        "GT_count": total_gt,
    }


## 3. Hàm Đọc file Tracks và Hàm Nội suy (Interpolation)

In [ ]:
def read_predictions_txt(txt_path: str) -> Dict[int, List[tuple]]:
    """
    Đọc kết quả tracking từ file text định dạng chuẩn MOT.
    Mỗi dòng: <frame>,<id>,<left>,<top>,<width>,<height>,...
    Trả về: {frame_num: [(track_id, [x1, y1, x2, y2]), ...]}
    """
    tracks_by_frame = {}
    if not os.path.exists(txt_path):
        return tracks_by_frame
    with open(txt_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(",")
            if len(parts) < 6:
                parts = line.split()
                if len(parts) < 6:
                    continue
            frame = int(float(parts[0]))
            track_id = int(float(parts[1]))
            left = float(parts[2])
            top = float(parts[3])
            width = float(parts[4])
            height = float(parts[5])

            x1 = left
            y1 = top
            x2 = left + width
            y2 = top + height

            if frame not in tracks_by_frame:
                tracks_by_frame[frame] = []
            tracks_by_frame[frame].append(
                (track_id, np.array([x1, y1, x2, y2], dtype=float))
            )
    return tracks_by_frame


def interpolate_tracks(start_tracks, end_tracks, alpha: float):
    """Nội suy tuyến tính vị trí của các xe giữa 2 frame kế cận để đánh giá đủ 100% số frame."""
    start_by_id = {track_id: bbox for track_id, bbox in start_tracks}
    end_by_id = {track_id: bbox for track_id, bbox in end_tracks}
    tracks = []

    for track_id in sorted(start_by_id.keys() & end_by_id.keys()):
        bbox = (1.0 - alpha) * start_by_id[track_id] + alpha * end_by_id[track_id]
        tracks.append((track_id, bbox))

    if alpha <= 0.0:
        for track_id in sorted(start_by_id.keys() - end_by_id.keys()):
            tracks.append((track_id, start_by_id[track_id]))
    elif alpha >= 1.0:
        for track_id in sorted(end_by_id.keys() - start_by_id.keys()):
            tracks.append((track_id, end_by_id[track_id]))
    return tracks


def build_eval_predictions(
    tracks_by_frame: Dict[int, List[tuple]], use_interpolation: bool, max_gap: int
):
    """Xây dựng danh sách các frame đánh giá, tự động chèn frame nội suy nếu cần."""
    frame_nums_sorted = sorted(tracks_by_frame.keys())
    if not frame_nums_sorted:
        return [], []

    pred_by_frame_eval = {}

    for idx, f_num in enumerate(frame_nums_sorted):
        pred_by_frame_eval[f_num] = tracks_by_frame[f_num]

        if not use_interpolation or idx + 1 >= len(frame_nums_sorted):
            continue

        next_f_num = frame_nums_sorted[idx + 1]
        gap = next_f_num - f_num

        if 1 < gap <= max_gap:
            for step in range(1, gap):
                alpha = step / gap
                pred_by_frame_eval[f_num + step] = interpolate_tracks(
                    tracks_by_frame[f_num], tracks_by_frame[next_f_num], alpha
                )

    eval_frame_nums = sorted(pred_by_frame_eval.keys())
    pred_frames = [pred_by_frame_eval[f] for f in eval_frame_nums]
    return eval_frame_nums, pred_frames


## 4. Cấu hình Đánh giá Metrics

In [ ]:
# === CẤU HÌNH ĐƯỜNG DẪN KAGGLE ===

# 1. Thư mục chứa các file .txt kết quả tracking của thuật toán cần test
PREDICTIONS_TRACKS_FOLDER = "/kaggle/working/sort_predictions"

# 2. Thư mục chứa XML Annotations Test Ground-Truth
ANNOTATIONS_FOLDER = "/kaggle/input/datasets/bratjay/ua-detrac-orig/DETRAC-Test-Annotations-XML/DETRAC-Test-Annotations-XML"

# --- THAM SỐ ĐÁNH GIÁ ---
LABEL_FRAME_STRIDE = (
    4  # Nếu tracker dự đoán cách nhau 4 frame thì giữ nguyên để nội suy
)
MOTA_EVALUATE_INTERPOLATED_FRAMES = True  # Có đánh giá các frame nội suy ở giữa không
MOTA_IOU_THRESHOLD = 0.50  # Ngưỡng IoU tiêu chuẩn

# --- LỌC VÙNG IGNORE KHI ĐÁNH GIÁ ---
FILTER_IGNORED_REGIONS = True
IGNORE_OVERLAP_THRESHOLD = 0.50

print("✓ Thiết lập cấu hình đánh giá thành công!")
print(f"Thư mục Tracking cần test: {PREDICTIONS_TRACKS_FOLDER}")
print(f"Thư mục XML Ground-Truth: {ANNOTATIONS_FOLDER}")

# --- CẤU HÌNH VẼ BIỂU ĐỒ HOTA ---
PLOT_TITLE = "SORT - Vehicle"


## 5. Khởi chạy Đánh giá và Xuất Bảng Kết quả MOTA

In [ ]:
pred_folder_path = Path(PREDICTIONS_TRACKS_FOLDER)
ann_folder_path = Path(ANNOTATIONS_FOLDER)

if not pred_folder_path.is_dir():
    raise FileNotFoundError(
        f"Thư mục chứa kết quả tracking không tồn tại: {pred_folder_path}"
    )
if not ann_folder_path.is_dir():
    raise FileNotFoundError(
        f"Thư mục XML Ground-truth không tồn tại: {ann_folder_path}"
    )

# Lấy danh sách các file tracking dự đoán
track_files = sorted(list(pred_folder_path.glob("*.txt")))
print(f"[*] Đang đánh giá {len(track_files)} file tracking kết quả...")

all_gt_sequences = {}
all_pred_sequences = {}
rows_metrics = []
overall_counts = {"FP": 0, "FN": 0, "ID_switches": 0, "GT_count": 0}

for t_file in tqdm(track_files, desc="Evaluating Sequences"):
    seq_name = t_file.stem  # Tên sequence tương ứng (ví dụ: MVI_39031)

    # Tải XML Ground-truth và Ignored region tương ứng
    xml_path = find_annotation_xml(seq_name, ANNOTATIONS_FOLDER)
    if not xml_path:
        print(f"⚠️ Không tìm thấy XML ground-truth cho sequence: {seq_name}. Bỏ qua.")
        continue
    gt_by_frame = load_detrac_gt_xml(xml_path)
    ignored_regions = load_detrac_ignored_regions_xml(xml_path)

    # Đọc kết quả tracking từ file text
    tracks_by_frame = read_predictions_txt(str(t_file))

    # Xây dựng danh sách đánh giá & nội suy
    eval_frame_nums, pred_frames = build_eval_predictions(
        tracks_by_frame,
        use_interpolation=MOTA_EVALUATE_INTERPOLATED_FRAMES,
        max_gap=LABEL_FRAME_STRIDE,
    )

    # Lọc bỏ các xe nằm đè lên vùng Ignore của sequence
    if FILTER_IGNORED_REGIONS:
        pred_frames = [
            filter_tracks_by_ignore(preds, ignored_regions, IGNORE_OVERLAP_THRESHOLD)
            for preds in pred_frames
        ]

    # Chuẩn bị ground-truth cho các frame đánh giá
    gt_frames = [gt_by_frame.get(f, []) for f in eval_frame_nums]

    # Ghi kết quả vào folder data/ dạng TrackEval
    data_folder = Path("data")
    data_folder.mkdir(parents=True, exist_ok=True)
    trackeval_file_path = data_folder / f"{seq_name}.txt"
    with open(trackeval_file_path, "w", encoding="utf-8") as trackeval_file:
        for frame_count in sorted(tracks_by_frame.keys()):
            for track_id, bbox in tracks_by_frame[frame_count]:
                x1 = bbox[0]
                y1 = bbox[1]
                w = bbox[2] - bbox[0]
                h = bbox[3] - bbox[1]
                score = 1.0
                track_class_id = 1
                trackeval_file.write(
                    f"{frame_count},{track_id},{x1},{y1},{w},{h},{score:.6f},{track_class_id},-1,-1\n"
                )

    # Lưu để vẽ biểu đồ HOTA
    all_gt_sequences[seq_name] = gt_frames
    all_pred_sequences[seq_name] = pred_frames

    # Tính toán chỉ số MOTA cho sequence
    metrics = compute_mota(gt_frames, pred_frames, iou_threshold=MOTA_IOU_THRESHOLD)

    # Cộng dồn chỉ số tổng thể
    for key in overall_counts:
        overall_counts[key] += metrics[key]

    # Lưu dòng dữ liệu của sequence
    rows_metrics.append(
        {
            "Sequence": seq_name,
            "MOTA (%)": metrics["MOTA"] * 100 if not np.isnan(metrics["MOTA"]) else 0.0,
            "GT Objects": metrics["GT_count"],
            "FP (False Positives)": metrics["FP"],
            "FN (False Negatives)": metrics["FN"],
            "ID Switches": metrics["ID_switches"],
            "Processed Frames": len(eval_frame_nums),
        }
    )

# Hiển thị kết quả bằng Pandas DataFrame
if rows_metrics:
    df = pd.DataFrame(rows_metrics)

    # Tạo hàng Tổng kết (Overall)
    overall_mota = (
        1.0
        - (overall_counts["FP"] + overall_counts["FN"] + overall_counts["ID_switches"])
        / overall_counts["GT_count"]
        if overall_counts["GT_count"] > 0
        else 0.0
    )

    overall_row = {
        "Sequence": "★ OVERALL (Tổng thể)",
        "MOTA (%)": overall_mota * 100,
        "GT Objects": overall_counts["GT_count"],
        "FP (False Positives)": overall_counts["FP"],
        "FN (False Negatives)": overall_counts["FN"],
        "ID Switches": overall_counts["ID_switches"],
        "Processed Frames": df["Processed Frames"].sum(),
    }

    df = pd.concat([df, pd.DataFrame([overall_row])], ignore_index=True)

    # Hiển thị bảng dạng rất trực quan
    print("\n" + "=" * 95)
    print("                      UA-DETRAC MULTIPLE OBJECT TRACKING (MOT) METRICS")
    print("=" * 95)

    pd.options.display.float_format = "{:,.2f}".format
    display(df)

    print(
        f"\n📝 Chế độ đánh giá: {'Đánh giá toàn bộ các frame (chèn nội suy tuyến tính)' if MOTA_EVALUATE_INTERPOLATED_FRAMES else 'Chỉ đánh giá các keyframe có nhãn'}"
    )
    print(f"🎯 Ngưỡng IoU tối thiểu để tính trùng khớp: {MOTA_IOU_THRESHOLD}")
    print(
        f"🚗 Lọc vùng Ignore: {'Có bật (Ngưỡng đè: ' + str(IGNORE_OVERLAP_THRESHOLD) + ')' if FILTER_IGNORED_REGIONS else 'Tắt'}"
    )
    print("=" * 95)
else:
    print("❌ Không tính toán được metrics nào. Hãy kiểm tra lại thư mục hoặc file.")


## 6. Tính toán HOTA và Vẽ Biểu đồ So sánh các Chỉ số


In [ ]:
import matplotlib.pyplot as plt
from collections import Counter


def compute_hota_metrics_for_dataset(gt_dataset, pred_dataset, alphas):
    results_by_alpha = {}

    seq_gt_counts = {}
    seq_pred_counts = {}

    for seq_name in gt_dataset.keys():
        gt_frames = gt_dataset[seq_name]
        pred_frames = pred_dataset.get(seq_name, [[] for _ in range(len(gt_frames))])

        gt_id_list = []
        for frame in gt_frames:
            for obj in frame:
                gt_id_list.append(obj[0])
        seq_gt_counts[seq_name] = Counter(gt_id_list)

        pred_id_list = []
        for frame in pred_frames:
            for obj in frame:
                pred_id_list.append(obj[0])
        seq_pred_counts[seq_name] = Counter(pred_id_list)

    for alpha in alphas:
        total_tps = 0
        total_fps = 0
        total_fns = 0

        all_tp_matches = []

        for seq_name in gt_dataset.keys():
            gt_frames = gt_dataset[seq_name]
            pred_frames = pred_dataset.get(
                seq_name, [[] for _ in range(len(gt_frames))]
            )

            for gt_objs, pred_objs in zip(gt_frames, pred_frames):
                gt_ids = [obj[0] for obj in gt_objs]
                gt_bboxes = [obj[1] for obj in gt_objs]
                pred_ids = [obj[0] for obj in pred_objs]
                pred_bboxes = [obj[1] for obj in pred_objs]

                matches, unmatched_gts, unmatched_preds = _match_gt_to_preds(
                    gt_bboxes, pred_bboxes, alpha
                )

                total_tps += len(matches)
                total_fps += len(unmatched_preds)
                total_fns += len(unmatched_gts)

                for g_idx, p_idx in matches:
                    gt_id = gt_ids[g_idx]
                    pred_id = pred_ids[p_idx]
                    match_iou = iou(gt_bboxes[g_idx], pred_bboxes[p_idx])
                    all_tp_matches.append((seq_name, gt_id, pred_id, match_iou))

        # Calculate association metrics
        seq_tpa_counts = {}
        for seq_name, gt_id, pred_id, match_iou in all_tp_matches:
            if seq_name not in seq_tpa_counts:
                seq_tpa_counts[seq_name] = Counter()
            seq_tpa_counts[seq_name][(gt_id, pred_id)] += 1

        sum_ass_iou = 0.0
        sum_ass_re = 0.0
        sum_ass_pr = 0.0
        sum_loc_iou = 0.0

        for seq_name, gt_id, pred_id, match_iou in all_tp_matches:
            tpa = seq_tpa_counts[seq_name][(gt_id, pred_id)]
            g_count = seq_gt_counts[seq_name][gt_id]
            p_count = seq_pred_counts[seq_name][pred_id]

            ass_iou = tpa / (g_count + p_count - tpa)
            ass_re = tpa / g_count
            ass_pr = tpa / p_count

            sum_ass_iou += ass_iou
            sum_ass_re += ass_re
            sum_ass_pr += ass_pr
            sum_loc_iou += match_iou

        if total_tps > 0:
            det_a = total_tps / (total_tps + total_fps + total_fns)
            det_re = total_tps / (total_tps + total_fns)
            det_pr = total_tps / (total_tps + total_fps)

            ass_a = sum_ass_iou / total_tps
            ass_re = sum_ass_re / total_tps
            ass_pr = sum_ass_pr / total_tps

            hota = np.sqrt(det_a * ass_a)
            loc_a = sum_loc_iou / total_tps
        else:
            det_a = 0.0
            det_re = 0.0
            det_pr = 0.0
            ass_a = 0.0
            ass_re = 0.0
            ass_pr = 0.0
            hota = 0.0
            loc_a = 0.0

        results_by_alpha[alpha] = {
            "HOTA": hota,
            "DetA": det_a,
            "AssA": ass_a,
            "DetRe": det_re,
            "DetPr": det_pr,
            "AssRe": ass_re,
            "AssPr": ass_pr,
            "LocA": loc_a,
        }

    return results_by_alpha


# Chạy tính toán HOTA trên các alpha
alphas = np.linspace(0.05, 0.95, 19)
print("[*] Đang tính toán các chỉ số HOTA, DetA, AssA... trên 19 ngưỡng alpha...")
hota_results = compute_hota_metrics_for_dataset(
    all_gt_sequences, all_pred_sequences, alphas
)
print("✓ Tính toán HOTA hoàn tất!")

# Trích xuất các list để vẽ đồ thị
alpha_list = sorted(hota_results.keys())
hota_list = [hota_results[a]["HOTA"] for a in alpha_list]
deta_list = [hota_results[a]["DetA"] for a in alpha_list]
assa_list = [hota_results[a]["AssA"] for a in alpha_list]
detre_list = [hota_results[a]["DetRe"] for a in alpha_list]
detpr_list = [hota_results[a]["DetPr"] for a in alpha_list]
assre_list = [hota_results[a]["AssRe"] for a in alpha_list]
asspr_list = [hota_results[a]["AssPr"] for a in alpha_list]
loca_list = [hota_results[a]["LocA"] for a in alpha_list]


# Hàm format giá trị trung bình để hiển thị trong legend giống ảnh của bạn
def fmt_mean(lst):
    val = np.mean(lst)
    return f"{val:.2f}".rstrip("0").rstrip(".")


mean_hota = fmt_mean(hota_list)
mean_deta = fmt_mean(deta_list)
mean_assa = fmt_mean(assa_list)
mean_detre = fmt_mean(detre_list)
mean_detpr = fmt_mean(detpr_list)
mean_assre = fmt_mean(assre_list)
mean_asspr = fmt_mean(asspr_list)
mean_loca = fmt_mean(loca_list)

# Vẽ biểu đồ
plt.figure(figsize=(8, 6))
plt.plot(alpha_list, hota_list, "r-", label=f"HOTA ({mean_hota})", linewidth=1.5)
plt.plot(alpha_list, deta_list, "b-", label=f"DetA ({mean_deta})", linewidth=1.5)
plt.plot(alpha_list, assa_list, "g-", label=f"AssA ({mean_assa})", linewidth=1.5)
plt.plot(alpha_list, detre_list, "b--", label=f"DetRe ({mean_detre})", linewidth=1.5)
plt.plot(alpha_list, detpr_list, "b:", label=f"DetPr ({mean_detpr})", linewidth=1.5)
plt.plot(alpha_list, assre_list, "g--", label=f"AssRe ({mean_assre})", linewidth=1.5)
plt.plot(alpha_list, asspr_list, "g:", label=f"AssPr ({mean_asspr})", linewidth=1.5)
plt.plot(alpha_list, loca_list, "m-", label=f"LocA ({mean_loca})", linewidth=1.5)

plt.title(PLOT_TITLE)
plt.xlabel("alpha")
plt.ylabel("score")
plt.xlim(0.0, 1.0)
plt.ylim(0.0, 1.0)
plt.legend(loc="lower left")
plt.grid(False)
plt.tight_layout()
plt.savefig("hota_metrics_plot.png", dpi=300)
plt.show()
